In [20]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os
from category_encoders import TargetEncoder


In [21]:
import pandas as pd
from typing import List, Tuple

def load_and_combine_months(file_paths: List[str]) -> pd.DataFrame:
    """
    Load multiple monthly parquet files and combine them into one DataFrame.
    Adds a 'month' column for time-based splitting.
    """
    dfs = []
    
    for path in file_paths:
        print(f"Loading: {path}")
        df = pd.read_parquet(path)
        
        # استخراج ماه از نام فایل (فرض: نام فایل شامل تاریخ است)
        # مثال: yellow_tripdata_2026-01.parquet → month = 1
        month = int(path.split('-')[-1].split('.')[0].split('_')[0])

        df['month'] = month
        
        dfs.append(df)
    
    combined = pd.concat(dfs, ignore_index=True)
    print(f"Total combined data shape: {combined.shape}")
    return combined


def split_train_validation_by_month(
    df: pd.DataFrame, 
    train_months: List[int], 
    val_month: int
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split data into Train and Validation based on month.
    """
    train_df = df[df['month'].isin(train_months)].copy()
    val_df = df[df['month'] == val_month].copy()
    
    print(f"Train shape: {train_df.shape} | Months: {train_months}")
    print(f"Validation shape: {val_df.shape} | Month: {val_month}")
    
    return train_df, val_df

In [22]:
# مسیر فایل‌ها
file_paths = [
    "/Users/mohsen/Desktop/nyc_duration_prediction/data/processed/yellow_tripdata_2026-01_processed.parquet",
    "/Users/mohsen/Desktop/nyc_duration_prediction/data/processed/yellow_tripdata_2026-02_processed.parquet",
    "/Users/mohsen/Desktop/nyc_duration_prediction/data/processed/yellow_tripdata_2026-03_processed.parquet",
]

# بارگذاری و ترکیب
df = load_and_combine_months(file_paths)

# تقسیم زمانی
train, val = split_train_validation_by_month(
    df=df,
    train_months=[1, 2],   # ماه ۱ و ۲ برای Train
    val_month=3            # ماه ۳ برای Validation
)


Loading: /Users/mohsen/Desktop/nyc_duration_prediction/data/processed/yellow_tripdata_2026-01_processed.parquet
Loading: /Users/mohsen/Desktop/nyc_duration_prediction/data/processed/yellow_tripdata_2026-02_processed.parquet
Loading: /Users/mohsen/Desktop/nyc_duration_prediction/data/processed/yellow_tripdata_2026-03_processed.parquet
Total combined data shape: (7748178, 28)
Train shape: (4838269, 28) | Months: [1, 2]
Validation shape: (2909909, 28) | Month: 3


In [23]:
train_df = train.sample(n=5000, random_state = 42).reset_index(drop=True)
val_df = val.sample(n=500, random_state = 42).reset_index(drop=True)
train_df

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,Airport_fee,cbd_congestion_fee,duration,pickup_hour,pickup_dayofweek,is_weekend,hour_category,is_rush_hour,PU_DO,month
0,2,2026-02-25 15:37:27,2026-02-25 15:49:43,1.0,1.41,1.0,N,142,161,1,...,0.0,0.75,12.27,15,2,False,midday,False,142_161,2
1,2,2026-01-11 02:16:38,2026-01-11 02:42:07,1.0,5.49,1.0,N,148,188,1,...,0.0,0.75,25.48,2,6,True,late_night,False,148_188,1
2,2,2026-02-14 00:41:06,2026-02-14 00:44:53,1.0,0.48,1.0,N,230,162,1,...,0.0,0.75,3.78,0,5,True,late_night,False,230_162,2
3,2,2026-02-02 14:02:25,2026-02-02 14:15:31,2.0,0.66,1.0,N,162,100,2,...,0.0,0.75,13.10,14,0,False,midday,False,162_100,2
4,1,2026-01-04 13:24:35,2026-01-04 13:42:08,1.0,1.50,1.0,N,163,186,1,...,0.0,0.75,17.55,13,6,True,midday,False,163_186,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,1,2026-01-08 11:47:59,2026-01-08 11:58:42,1.0,1.60,1.0,N,41,166,1,...,0.0,0.00,10.72,11,3,False,midday,False,41_166,1
4996,2,2026-01-02 00:39:28,2026-01-02 00:57:12,1.0,6.87,1.0,N,161,244,1,...,0.0,0.75,17.73,0,4,False,late_night,False,161_244,1
4997,2,2026-01-30 20:29:32,2026-01-30 20:53:20,1.0,2.18,1.0,N,79,186,1,...,0.0,0.75,23.80,20,4,False,night,False,79_186,1
4998,2,2026-01-31 09:19:27,2026-01-31 09:27:49,2.0,2.09,1.0,N,90,231,1,...,0.0,0.75,8.37,9,5,True,morning_peak,True,90_231,1


In [24]:
def select_features(df: pd.DataFrame):
    """Select features and define categorical vs numerical."""
    categorical = ['hour_category']           # فقط این را وکتورایز می‌کنیم (تعداد دسته کم است)
    
    numerical = [
        'trip_distance',
        'PULocationID',
        'DOLocationID',
        'pickup_dayofweek',
        'pickup_hour',
        'is_rush_hour',
        'is_weekend',
        'PU_DO'
    ]
    target = 'duration'

    
    df = df[categorical + numerical + [target]].copy()
    return df, categorical, numerical, target


In [25]:
# ========================
# انتخاب فیچرها
# ========================
train_df, categorical, numerical, target = select_features(train_df)
val_df, _, _, _ = select_features(val_df)
print(train_df)
# ========================
# Target Encoding برای PU_DO (حرفه‌ای)
# ========================
target_encoder = TargetEncoder(
    cols=['PU_DO'], 
    smoothing=15, 
    min_samples_leaf=30
)

# فقط روی داده Train فیت می‌کنیم
target_encoder.fit(train_df[['PU_DO']], train_df[target])

train_df['PU_DO_encoded'] = target_encoder.transform(train_df[['PU_DO']])
val_df['PU_DO_encoded'] = target_encoder.transform(val_df[['PU_DO']])

# ========================
# فیچرهای نهایی برای مدل
# ========================
# PU_DO را حذف می‌کنیم و نسخه encode شده‌اش را نگه می‌داریم
final_categorical = ['hour_category']                    # فقط این را وان‌هات می‌کنیم
final_numerical = ['trip_distance', 'pickup_dayofweek', 'PU_DO_encoded']

# ========================
# وکتورایز کردن (فقط hour_category)
# ========================
dv = DictVectorizer()

train_dicts = train_df[final_categorical + final_numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = val_df[final_categorical + final_numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

y_train = train_df[target].values
y_val = val_df[target].values

     hour_category  trip_distance  PULocationID  DOLocationID  \
0           midday           1.41           142           161   
1       late_night           5.49           148           188   
2       late_night           0.48           230           162   
3           midday           0.66           162           100   
4           midday           1.50           163           186   
...            ...            ...           ...           ...   
4995        midday           1.60            41           166   
4996    late_night           6.87           161           244   
4997         night           2.18            79           186   
4998  morning_peak           2.09            90           231   
4999        midday           0.81           186            90   

      pickup_dayofweek  pickup_hour  is_rush_hour  is_weekend    PU_DO  \
0                    2           15         False       False  142_161   
1                    6            2         False        True  148_188 

In [34]:
X_train.shape

(5000, 8)

In [28]:
import mlflow 
mlflow.set_tracking_uri("http://127.0.0.1:5000")
print("Tracking URI:", mlflow.get_tracking_uri())
mlflow.set_experiment('nyc_duration_prediction_boosting_model')

Tracking URI: http://127.0.0.1:5000


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1785662749491, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1785662749491, lifecycle_stage='active', name='nyc_duration_prediction_boosting_model', tags={}, trace_location=None, workspace='default'>

In [29]:
import mlflow
import optuna
import lightgbm as lgb
import joblib
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from mlflow.models import infer_signature


# ========================
# تابع هدف (Objective)
# ========================
def objective(trial, X_train, X_val, y_train, y_val):
    with mlflow.start_run(nested=True, run_name=f"lgb_trial_{trial.number}"):

        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'n_estimators': trial.suggest_int('n_estimators', 300, 2000),
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state': 42,
            'n_jobs': 2,
            'verbose': -1
        }

        model = lgb.LGBMRegressor(**params)

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)

        mlflow.log_params(params)
        mlflow.log_metric("rmse", rmse)

        return rmse


# ========================
# تابع اصلی اجرای Tuning
# ========================
def run_lightgbm_optuna_tuning(X_train, X_val, y_train, y_val, n_trials=50):
    
    with mlflow.start_run(run_name="lightgbm_optuna_tuning_version_8"):
        
        study = optuna.create_study(
            direction="minimize",
            study_name="LightGBM_Hyperparameter_Tuning",
            pruner=optuna.pruners.MedianPruner()
        )

        study.optimize(
            lambda trial: objective(trial, X_train, X_val, y_train, y_val),
            n_trials=n_trials,
            show_progress_bar=True
        )

        print("\n" + "="*60)
        print(f"بهترین RMSE: {study.best_value:.4f}")
        for k, v in study.best_params.items():
            print(f"  {k}: {v}")
        print("="*60)

        mlflow.log_params(study.best_params)
        mlflow.log_metric("best_rmse", study.best_value)

        # ========================
        # آموزش مدل نهایی
        # ========================
        best_params = study.best_params.copy()
        best_params.update({
            'objective': 'regression',
            'metric': 'rmse',
            'random_state': 42,
            'n_jobs': 2,
            'verbose': -1
        })

        final_model = lgb.LGBMRegressor(**best_params)

        final_model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        # ارزیابی نهایی
        y_pred = final_model.predict(X_val)
        final_rmse = root_mean_squared_error(y_val, y_pred)
        final_mae = mean_absolute_error(y_val, y_pred)
        final_r2 = r2_score(y_val, y_pred)

        mlflow.log_metrics({
            "final_rmse": final_rmse,
            "final_mae": final_mae,
            "final_r2": final_r2
        })

        # ========================
        # ذخیره Artifactهای پیش‌پردازش (خیلی مهم)
        # ========================
        # ذخیره TargetEncoder
        joblib.dump(target_encoder, "target_encoder.pkl")
        mlflow.log_artifact("target_encoder.pkl", artifact_path="preprocessing")

        # ذخیره DictVectorizer
        joblib.dump(dv, "dict_vectorizer.pkl")
        mlflow.log_artifact("dict_vectorizer.pkl", artifact_path="preprocessing")

        # ========================
        # لاگ کردن مدل (با Signature و Metadata)
        # ========================
        signature = infer_signature(X_train, y_train)

        mlflow.sklearn.log_model(
            sk_model=final_model,
            artifact_path="model",
            signature=signature,
            skops_trusted_types=[
                "lightgbm.sklearn.LGBMRegressor",
                "lightgbm.basic.Booster",
                "collections.OrderedDict"
            ],
            metadata={
                "description": "LightGBM model trained with Optuna (50 trials)",
                "training_period": "AUG 2026",
                "feature_engineering": "Target Encoding on PU_DO + One-Hot on hour_category",
                "number_of_trials": 50,
                "best_rmse": round(final_rmse, 4),
                "n_jobs": 2,
                "preprocessing_artifacts": "target_encoder.pkl, dict_vectorizer.pkl"
            }
        )
        print(f"نوع داده: {type(X_train)}")
        print(f"تعداد ردیف و ستون: {X_train.shape}")
        print("\n✅ مدل نهایی لاگ شد (همراه با Artifactهای پیش‌پردازش).")
        return final_model, study


# ========================
# اجرا
# ========================
if __name__ == "__main__":
    run_lightgbm_optuna_tuning(X_train, X_val, y_train, y_val, n_trials=50)

[I 2026-08-02 17:34:33,631] A new study created in memory with name: LightGBM_Hyperparameter_Tuning
  0%|                                                                                                         | 0/50 [00:00<?, ?it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 0. Best value: 11.311:   2%|█▏                                                            | 1/50 [00:00<00:08,  5.93it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


🏃 View run lgb_trial_0 at: http://127.0.0.1:5000/#/experiments/2/runs/62cb59615ab549c0b6ff6683ef11c738
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:34:33,801] Trial 0 finished with value: 11.310955556664164 and parameters: {'n_estimators': 620, 'num_leaves': 140, 'max_depth': 12, 'learning_rate': 0.19448795481382375, 'feature_fraction': 0.8426067791814889, 'bagging_fraction': 0.9601350039545333, 'bagging_freq': 4, 'min_child_samples': 60, 'reg_alpha': 0.00012950253127922017, 'reg_lambda': 0.00015761178349792717}. Best is trial 0 with value: 11.310955556664164.


Best trial: 0. Best value: 11.311:   4%|██▍                                                           | 2/50 [00:00<00:17,  2.82it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 0. Best value: 11.311:   4%|██▍                                                           | 2/50 [00:00<00:17,  2.82it/s]

🏃 View run lgb_trial_1 at: http://127.0.0.1:5000/#/experiments/2/runs/d5926c6a30d742ac93c575ceb7787fc4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:34:34,286] Trial 1 finished with value: 12.005369274568436 and parameters: {'n_estimators': 924, 'num_leaves': 23, 'max_depth': 8, 'learning_rate': 0.01660955220991357, 'feature_fraction': 0.9977154080369821, 'bagging_fraction': 0.8632428172354419, 'bagging_freq': 6, 'min_child_samples': 12, 'reg_alpha': 0.00039982595416673204, 'reg_lambda': 7.368611859587296e-07}. Best is trial 0 with value: 11.310955556664164.
🏃 View run lgb_trial_2 at: http://127.0.0.1:5000/#/experiments/2/runs/50347d3d88b740fe9ac02af6c026a8c4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:34:34,347] Trial 2 finished with value: 11.403529371276578 and parameters: {'n_estimators': 1537, 'num_leaves': 93, 'max_depth': 3, 'learning_rate': 0.16671670695516266, 'feature_fraction': 0.9270317564735311, 'bagging_fract

/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 3. Best value: 10.8791:   8%|████▉                                                        | 4/50 [00:00<00:09,  4.64it/s]

...

🏃 View run lgb_trial_46 at: http://127.0.0.1:5000/#/experiments/2/runs/e89510413c9146ffa86d232f65e92d59
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:34:42,198] Trial 46 finished with value: 10.435180718361481 and parameters: {'n_estimators': 1323, 'num_leaves': 120, 'max_depth': 11, 'learning_rate': 0.2991130148478517, 'feature_fraction': 0.6231329981319658, 'bagging_fraction': 0.7326787355413836, 'bagging_freq': 4, 'min_child_samples': 36, 'reg_alpha': 0.002962364306026666, 'reg_lambda': 5.095908304033133e-07}. Best is trial 43 with value: 10.38484370964625.
🏃 View run lgb_trial_47 at: http://127.0.0.1:5000/#/experiments/2/runs/aec0c5f4029c48aea0109880dbfc5517
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:34:42,319] Trial 47 finished with value: 10.461490152857372 and parameters: {'n_estimators': 1332, 'num_leaves': 121, 'max_depth': 11, 'learning_rate': 0.2967085040237171, 'feature_fraction': 0.6515466302972415, 'bagging_

/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 43. Best value: 10.3848:  98%|█████████████████████████████████████████████████████████▊ | 49/50 [00:08<00:00,  7.37it/s]/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 43. Best value: 10.3848: 100%|███████████████████████████████████████████████████████████| 50/50 [00:08<00:00,  5.62it/s]


🏃 View run lgb_trial_48 at: http://127.0.0.1:5000/#/experiments/2/runs/ed167ae7b9a84d138c3b42b50e8ebc60
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:34:42,470] Trial 48 finished with value: 10.572764420294572 and parameters: {'n_estimators': 1462, 'num_leaves': 131, 'max_depth': 11, 'learning_rate': 0.25026924751046564, 'feature_fraction': 0.6201691843556278, 'bagging_fraction': 0.7386936404740785, 'bagging_freq': 2, 'min_child_samples': 30, 'reg_alpha': 0.14590772464498084, 'reg_lambda': 6.90794863524122e-08}. Best is trial 43 with value: 10.38484370964625.
🏃 View run lgb_trial_49 at: http://127.0.0.1:5000/#/experiments/2/runs/92e613334d2b4e29aed7ad7f31f9e38c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-08-02 17:34:42,531] Trial 49 finished with value: 11.413795832872147 and parameters: {'n_estimators': 1914, 'num_leaves': 117, 'max_depth': 3, 'learning_rate': 0.21676643419341368, 'feature_fraction': 0.7117742724022141, 'bagging_f

/Users/mohsen/Desktop/nyc_duration_prediction/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
2026/08/02 17:34:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/02 17:34:47 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


نوع داده: <class 'scipy.sparse._csr.csr_matrix'>
تعداد ردیف و ستون: (5000, 8)

✅ مدل نهایی لاگ شد (همراه با Artifactهای پیش‌پردازش).
🏃 View run lightgbm_optuna_tuning_version_8 at: http://127.0.0.1:5000/#/experiments/2/runs/d27a0a47d6094e34b85ee2c1abff8f66
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [30]:
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature

mlflow.set_tracking_uri("http://127.0.0.1:5000")
client = MlflowClient()

model_name = "NYC_Taxi_Duration_LightGBM_version_8"

# ========================
# پیدا کردن بهترین Run
# ========================
runs = client.search_runs(
    experiment_ids=["2"],   # شماره Experiment خودت را بگذار
    filter_string="tags.mlflow.runName = 'lightgbm_optuna_tuning_version_8'",
    order_by=["metrics.best_rmse ASC"],
    max_results=1
)

if not runs:
    print("Run پیدا نشد!")
else:
    best_run = runs[0]
    run_id = best_run.info.run_id
    best_rmse = best_run.data.metrics.get("best_rmse")

    print(f"بهترین Run پیدا شد: {run_id}")
    print(f"بهترین RMSE: {best_rmse}")

    # ========================
    # ثبت مدل در Registry
    # ========================
    model_version = mlflow.register_model(
        model_uri=f"runs:/{run_id}/model",
        name=model_name
    )

    print(f"مدل ثبت شد → Version: {model_version.version}")

    # ========================
    # اضافه کردن توضیحات (Description)
    # ========================
    description = f"""
LightGBM model trained with Optuna hyperparameter tuning.

- Training Period: January & February 2026
- Best Validation RMSE: {round(best_rmse, 4)}
- Number of Optuna Trials: 50
- Feature Engineering: Target Encoding on PU_DO
- Final Features: trip_distance, pickup_dayofweek, PU_DO_encoded, hour_category (One-Hot)
- n_jobs: 2
"""

    client.update_model_version(
        name=model_name,
        version=model_version.version,
        description=description.strip()
    )

    # ========================
    # اضافه کردن تگ‌ها (Tags)
    # ========================
    tags = {
        "project": "NYC_Taxi_Duration",
        "team": "Data Science",
        "model_type": "LightGBM",
        "tuning_method": "Optuna",
        "training_period": "Jan-Feb 2026",
        "best_rmse": str(round(best_rmse, 4)),
        "number_of_trials": "50",
        "feature_engineering": "Target Encoding",
        "created_by": "Mohsen"
    }

    for key, value in tags.items():
        client.set_model_version_tag(
            name=model_name,
            version=model_version.version,
            key=key,
            value=value
        )

    # ========================
    # تنظیم Alias به عنوان Champion
    # ========================
    client.set_registered_model_alias(
        name=model_name,
        alias="champion",
        version=model_version.version
    )

    print(f"✅ مدل به عنوان @champion تنظیم شد (Version {model_version.version})")
    print("Description و Tags با موفقیت اضافه شدند.")

Successfully registered model 'NYC_Taxi_Duration_LightGBM_version_8'.
2026/08/02 17:35:43 WARNING mlflow.tracking._model_registry.fluent: Run with id d27a0a47d6094e34b85ee2c1abff8f66 has no artifacts at artifact path 'model', registering model based on models:/m-c45ef444b7f34cb188ba2c7e103f941f instead
2026/08/02 17:35:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: NYC_Taxi_Duration_LightGBM_version_8, version 1


بهترین Run پیدا شد: d27a0a47d6094e34b85ee2c1abff8f66
بهترین RMSE: 10.38484370964625
مدل ثبت شد → Version: 1


Created version '1' of model 'NYC_Taxi_Duration_LightGBM_version_8'.


✅ مدل به عنوان @champion تنظیم شد (Version 1)
Description و Tags با موفقیت اضافه شدند.


In [32]:
import mlflow
import pandas as pd
import joblib
from scipy.sparse import hstack

mlflow.set_tracking_uri("http://127.0.0.1:5000")

# ========================
# لود مدل و پیش‌پردازش
# ========================
model = mlflow.sklearn.load_model("models:/NYC_Taxi_Duration_LightGBM_version_8@champion")
target_encoder = joblib.load("target_encoder.pkl")
dv = joblib.load("dict_vectorizer.pkl")

print("✅ مدل و پیش‌پردازش‌ها لود شدند.")

# ========================
# داده نمونه
# ========================
df_real = pd.DataFrame({
    'hour_category': ['midday', 'late_night', 'late_night', 'midday', 'midday'],
    'trip_distance': [1.41, 5.49, 0.48, 0.66, 1.50],
    'PULocationID': [142, 148, 230, 162, 163],
    'DOLocationID': [161, 188, 162, 100, 186],
    'pickup_dayofweek': [2, 6, 5, 0, 6]
})

print("\n" + "="*60)
print("داده خام:")
print(df_real)
print("="*60)

# ========================
# مرحله ۱: ایجاد PU_DO + Target Encoding
# ========================
df_real['PU_DO'] = df_real['PULocationID'].astype(str) + '_' + df_real['DOLocationID'].astype(str)
df_real['PU_DO_encoded'] = target_encoder.transform(df_real[['PU_DO']])

print("\nبعد از Target Encoding:")
print(f"تعداد ستون df_real: {df_real.shape[1]}")
print(df_real.columns.tolist())

# ========================
# مرحله ۲: تبدیل hour_category به One-Hot
# ========================
cat_dicts = df_real[['hour_category']].to_dict(orient='records')
print(cat_dicts)
X_cat = dv.transform(cat_dicts)
X_cat_df = pd.DataFrame(X_cat.toarray(), columns=dv.get_feature_names_out())

print("\nبعد از One-Hot hour_category:")
print(f"تعداد ستون X_cat_df: {X_cat_df.shape[1]}")
print("ستون‌های X_cat_df:", X_cat_df.columns.tolist())

# ========================
# مرحله ۳: ساخت X_new با ترتیب درست (مهم!)
# ========================
# ترتیب صحیح (بر اساس خروجی قبلی X_train):
# ['PU_DO_encoded', hour cats (5 ستون), 'pickup_dayofweek', 'trip_distance']

pu_do_part = df_real[['PU_DO_encoded']].values
cat_part   = X_cat
day_part   = df_real[['pickup_dayofweek']].values
dist_part  = df_real[['trip_distance']].values

X_new = X_cat_df

print("\n" + "="*60)
print(f"تعداد فیچر نهایی X_new: {X_new.shape[1]}")
print("="*60)

# ========================
# پیش‌بینی
# ========================
predictions = model.predict(X_new)

df_real['predicted_duration'] = predictions.round(2)

print("\nنتایج پیش‌بینی:")
print(df_real[['trip_distance', 'PULocationID', 'DOLocationID', 'hour_category', 'predicted_duration']])

✅ مدل و پیش‌پردازش‌ها لود شدند.

داده خام:
  hour_category  trip_distance  PULocationID  DOLocationID  pickup_dayofweek
0        midday           1.41           142           161                 2
1    late_night           5.49           148           188                 6
2    late_night           0.48           230           162                 5
3        midday           0.66           162           100                 0
4        midday           1.50           163           186                 6

بعد از Target Encoding:
تعداد ستون df_real: 7
['hour_category', 'trip_distance', 'PULocationID', 'DOLocationID', 'pickup_dayofweek', 'PU_DO', 'PU_DO_encoded']
[{'hour_category': 'midday'}, {'hour_category': 'late_night'}, {'hour_category': 'late_night'}, {'hour_category': 'midday'}, {'hour_category': 'midday'}]

بعد از One-Hot hour_category:
تعداد ستون X_cat_df: 8
ستون‌های X_cat_df: ['PU_DO_encoded', 'hour_category=evening_peak', 'hour_category=late_night', 'hour_category=midday', 'hour_ca